# 05 — LLM Prompt Evaluation
Evaluates prompt quality across all 12 DataPilot prompts using MockLLMService
and a set of canonical test cases. Measures output format compliance,
parse success rate, and semantic quality without incurring Bedrock costs.

**Real-time integration**: prompt failures cause agent retries (each adds
~300ms–2s to the pipeline). This notebook identifies which prompts are most
fragile so we can harden them before production.


In [ ]:
import sys, json, re, time, asyncio
sys.path.insert(0, '..')

# Test cases: (prompt_name, input_variables, expected_output_shape)
INTENT_CASES = [
    ('total revenue by region',      {'intent': 'statistical_question', 'requires_sql': True,  'requires_rag': False}),
    ('forecast next quarter sales',  {'intent': 'forecasting_request',  'requires_sql': False, 'requires_forecast': True}),
    ('show me a revenue chart',      {'intent': 'visualization_request','requires_viz': True}),
    ('are there any data anomalies', {'intent': 'anomaly_investigation', 'requires_rag': True}),
    ('download the report as PDF',   {'intent': 'data_export'}),
]

SQL_CASES = [
    ('total revenue by region', 'SELECT "region", SUM("revenue")', True),
    ('top 5 products by quantity', 'ORDER BY', True),
    ('INSERT INTO orders VALUES', None, False),  # should be rejected
    ('DROP TABLE dataset', None, False),          # should be rejected
]

print("Test cases loaded. Running evaluation...")


## Intent classification — format compliance

In [ ]:
import pathlib

intent_prompt = pathlib.Path('../prompts/intent/v1.txt').read_text()

# Simulate LLM responses for the intent prompt
def mock_intent_response(query: str) -> str:
    """Generate a realistic intent classification JSON response."""
    routing = {
        'total revenue by region':     ('statistical_question', True,  False, False, False),
        'forecast next quarter sales':  ('forecasting_request',  False, True,  True,  False),
        'show me a revenue chart':      ('visualization_request', True,  False, False, True),
        'are there any data anomalies': ('anomaly_investigation', False, False, False, False),
        'download the report as PDF':   ('data_export',           False, False, False, False),
    }
    intent, sql, rag, forecast, viz = routing.get(query, ('general_question', False, True, False, False))
    return json.dumps({
        'intent': intent,
        'entities': {'column': None, 'metric': 'sum' if sql else None, 'time_range': None},
        'confidence': 0.92,
        'sub_intents': [],
        'requires_sql': sql,
        'requires_rag': rag,
        'requires_forecast': forecast,
        'requires_viz': viz,
        'search_query': f'{query} in the dataset',
    })

print("── Intent Classification Format Compliance ──\n")
all_pass = True
for query, expected in INTENT_CASES:
    raw      = mock_intent_response(query)
    try:
        parsed = json.loads(raw)
        # Check required fields
        required = ['intent', 'entities', 'confidence', 'requires_sql', 'requires_rag', 'search_query']
        missing  = [f for f in required if f not in parsed]
        valid_intent = parsed['intent'] in [
            'statistical_question','forecasting_request','anomaly_investigation',
            'data_export','general_question','sql_query','visualization_request',
            'comparison','trend_analysis'
        ]
        conf_valid = 0.0 <= parsed['confidence'] <= 1.0
        format_ok  = not missing and valid_intent and conf_valid

        # Check routing flags match expected
        route_ok = all(parsed.get(k) == v for k, v in expected.items() if k != 'intent')
        intent_ok = parsed['intent'] == expected.get('intent', parsed['intent'])

        status = '✓' if (format_ok and route_ok and intent_ok) else '✗'
        if status == '✗': all_pass = False
        print(f"  {status} '{query[:40]}'")
        if not format_ok: print(f"      Missing fields: {missing}")
        if not intent_ok: print(f"      Intent: expected={expected.get('intent')}, got={parsed['intent']}")
        if not route_ok:  print(f"      Routing mismatch")
    except json.JSONDecodeError as e:
        all_pass = False
        print(f"  ✗ '{query[:40]}' — JSON parse error: {e}")

print(f"\nResult: {'ALL PASS ✓' if all_pass else 'FAILURES DETECTED ✗'}")


## SQL validator — injection and DDL rejection

In [ ]:
from backend.agents.analysis.sql.sql_validator import validate, SQLValidationError, is_safe

sql_cases = [
    ("SELECT * FROM dataset LIMIT 10",                    True,  "basic select"),
    ('SELECT "region", SUM("revenue") FROM dataset GROUP BY "region"', True, "aggregation"),
    ("SELECT * FROM dataset WHERE revenue > 1000",         True,  "filter"),
    ("INSERT INTO dataset VALUES (1, 2, 3)",               False, "DML blocked"),
    ("DROP TABLE dataset",                                  False, "DDL blocked"),
    ("SELECT * FROM dataset; DROP TABLE dataset",          False, "multi-statement"),
    ("SELECT * FROM dataset -- union injection",            False, "comment injection"),
    ('SELECT * FROM READ_CSV("malicious.csv")',             False, "file access blocked"),
    ("```sql\nSELECT * FROM dataset\n```",               True,  "markdown fences stripped"),
    ("EXEC xp_cmdshell('rm -rf /')",                       False, "exec blocked"),
]

print("── SQL Validator — Safety Check Results ──\n")
pass_count = 0
for sql, should_pass, description in sql_cases:
    try:
        result   = validate(sql)
        passed   = True
        outcome  = f"ALLOWED: {result[:60]}..."
    except SQLValidationError as e:
        passed   = False
        outcome  = f"BLOCKED: {str(e)[:60]}"
    except Exception as e:
        passed   = False
        outcome  = f"ERROR: {str(e)[:60]}"

    correct = passed == should_pass
    if correct: pass_count += 1
    status  = '✓' if correct else '✗'
    print(f"  {status} [{description}]")
    print(f"      Expected: {'PASS' if should_pass else 'BLOCK'} | Got: {'PASS' if passed else 'BLOCK'}")
    if not correct:
        print(f"      {outcome}")

print(f"\n{pass_count}/{len(sql_cases)} correct ({pass_count/len(sql_cases):.0%})")


## Critic scoring — validates 5-criterion rubric

In [ ]:
def evaluate_insight_quality(insight: dict) -> dict:
    """Apply the 5-criterion rubric from the critic prompt."""
    scores = {}
    text = (insight.get('headline','') + ' ' + insight.get('explanation','')).lower()

    # 1. Accuracy: cites column name + number
    has_number = bool(re.search(r'\d+[%$.,]?\d*', text))
    has_column = any(col in text for col in ['revenue','region','quantity','discount','status','order'])
    scores['accuracy'] = 1.0 if (has_number and has_column) else 0.5 if has_column else 0.0

    # 2. Specificity: no vague words
    vague = ['some','many','various','significant','notable','considerable','substantial','quite','rather']
    scores['specificity'] = 0.0 if any(v in text for v in vague) else 1.0

    # 3. Relevance: actionable language
    action_words = ['suggests','indicates','team','consider','reduce','increase','improve','address','review']
    scores['relevance'] = 1.0 if any(w in text for w in action_words) else 0.5

    # 4. Completeness: what + why
    has_what = any(w in text for w in ['shows','reveals','found','detected','identified'])
    has_why  = any(w in text for w in ['suggests','means','impact','risk','opportunity','because'])
    scores['completeness'] = 1.0 if (has_what and has_why) else 0.5 if (has_what or has_why) else 0.0

    # 5. Bias: no absolute language
    absolutes = ['always','never','all orders','every','none','impossible','certain','proven']
    scores['bias'] = 0.0 if any(a in text for a in absolutes) else 1.0

    scores['overall'] = round(sum(scores.values()) / 5, 3)
    scores['passes']  = scores['overall'] >= 0.75 and scores['accuracy'] > 0.0
    return scores

test_insights = [
    {
        'headline': 'North region generates 31% of total revenue at $187,420 in 2024',
        'explanation': 'The revenue column shows North contributing $187,420 (31%) versus South at $142,800 (23%). This suggests the sales team should investigate what drives North region outperformance.',
        'expected_pass': True,
    },
    {
        'headline': 'Revenue has increased significantly over time',
        'explanation': 'There are many orders that show various revenue patterns across different regions.',
        'expected_pass': False,
    },
    {
        'headline': 'The discount_pct column always reduces revenue by a large amount',
        'explanation': 'Every discounted order shows lower revenue compared to full-price orders.',
        'expected_pass': False,
    },
]

print("── Insight Quality Scoring (5-Criterion Rubric) ──\n")
for ins in test_insights:
    scores = evaluate_insight_quality(ins)
    result = '✓ PASS' if scores['passes'] == ins['expected_pass'] else '✗ WRONG'
    print(f"  {result}  '{ins['headline'][:60]}'")
    print(f"    Scores: accuracy={scores['accuracy']:.1f} specificity={scores['specificity']:.1f} "
          f"relevance={scores['relevance']:.1f} completeness={scores['completeness']:.1f} bias={scores['bias']:.1f}")
    print(f"    Overall: {scores['overall']:.3f} | Passes: {scores['passes']} (expected: {ins['expected_pass']})\n")


## Prompt template variable coverage check

In [ ]:
# Verify all {variable} placeholders in prompt files are accounted for
import pathlib, re

prompt_vars = {
    'sql/v1.txt':           ['schema_summary', 'date_columns', 'numeric_columns', 'row_count', 'user_question'],
    'intent/v1.txt':        ['column_names', 'user_message'],
    'schema/v1.txt':        ['column_name', 'data_type', 'sample_values', 'null_rate', 'unique_count', 'total_rows'],
    'planner/v1.txt':       ['has_datetime_columns', 'numeric_column_count', 'row_count', 'trigger', 'dataset_id', 'schema_summary', 'plan_id'],
    'insight/v1.txt':       ['row_count', 'column_count', 'completeness_score', 'duplicate_count', 'column_names', 'sql_summary', 'anomaly_summary', 'forecast_summary', 'ml_summary', 'rag_context', 'previous_critique'],
    'critic/v1.txt':        ['profile_summary', 'insights_json'],
    'forecast/v1.txt':      ['target_column', 'date_column', 'horizon_label', 'model_name', 'trend_direction', 'training_rows', 'predictions_sample', 'predictions_end_sample', 'min_forecast', 'max_forecast', 'avg_forecast'],
    'recommendation/v1.txt':['row_count', 'column_count', 'column_names', 'insights_json', 'anomaly_count', 'forecast_trend', 'ml_top_features'],
    'validation/v1.txt':    ['profile_summary_json', 'response_text'],
    'python/v1.txt':        ['schema_summary', 'row_count', 'task_description'],
    'report/v1.txt':        ['format', 'dataset_name', 'row_count', 'column_count', 'executive_summary', 'insights_json', 'completeness_score', 'duplicate_count', 'anomaly_count', 'high_null_columns', 'recommendations_json'],
}

print("── Prompt Template Variable Coverage ──\n")
all_ok = True
for prompt_file, expected_vars in prompt_vars.items():
    path = pathlib.Path(f'../prompts/{prompt_file}')
    if not path.exists():
        print(f"  ✗ MISSING: {prompt_file}")
        all_ok = False
        continue
    content   = path.read_text()
    found_vars = re.findall(r'\{(\w+)\}', content)
    missing   = [v for v in expected_vars if v not in found_vars]
    extra     = [v for v in found_vars if v not in expected_vars]
    ok = not missing
    if not ok: all_ok = False
    status = '✓' if ok else '✗'
    print(f"  {status} {prompt_file}")
    if missing: print(f"      MISSING vars: {missing}")
    if extra:   print(f"      EXTRA vars:   {extra} (check if intentional)")

print(f"\n{'All prompts OK ✓' if all_ok else 'Prompt issues found ✗'}")
